# Módulo 04 · Projeto final: documentos novos na base

**Duração:** 1h30 (1 hora de notebook e 30 minutos com o app no ar)

## Enunciado

A TechNova publicou três documentos novos, que estão em `data/new_documents/`:

| Arquivo | Conteúdo |
|---|---|
| `nova_buds_firmware_update.md` | comunicado da versão 1.3.0 do firmware do Fone Nova Buds (corrige desconexões no multiponto) |
| `nova_cinema_soundbar.html` | artigo da central de ajuda sobre a Soundbar Nova Cinema (instalação, conexões, problemas) |
| `sm-005_nova_fold.pdf` | manual do Smartphone Nova Fold |

Sua missão é incorporar os documentos ao assistente **com evidência**: mostrar que a base passou a responder o que
não sabia, que a busca encontra os documentos novos e que **nada piorou** nas perguntas antigas. Só então a mudança
vai para produção.

A regra de ouro: **nunca altere o índice principal** (`data/faiss_index`). Todo o trabalho acontece em uma cópia,
em uma pasta temporária. No fim, você vê como tornar a inclusão permanente e rodar o app.

## Atividades

1. **Antes e depois**: perguntar sobre o firmware 1.3.0 do Nova Buds antes da inclusão (a base não sabe) e depois
   (responde citando o comunicado).
2. **Perguntas novas de avaliação**: criar 3 perguntas sobre os documentos novos e medir a recuperação antes e
   depois.
3. **Regressão**: rodar a avaliação completa de recuperação nas 26 perguntas antigas e garantir que nada piorou.
4. **Produção**: conferir que o índice principal está intacto e preparar os comandos para rodar o app e tornar a
   inclusão permanente.

## Critérios de sucesso

| # | Critério | Onde é verificado |
|---|---|---|
| 1 | Antes da inclusão, o assistente admite que não sabe sobre o firmware 1.3.0 | Verificação 1 |
| 2 | A cópia do índice recebe os trechos novos; o índice principal não muda | Verificação 2 |
| 3 | Depois da inclusão, a resposta cita `nova_buds_firmware_update.md` | Verificação 3 |
| 4 | 3 perguntas novas válidas: hit rate 0 antes e pelo menos 2/3 depois | Verificação 4 |
| 5 | Nenhuma pergunta antiga perde o acerto (hit) e o MRR não cai mais que 0,05 | Verificação 5 |
| 6 | Comando do app aponta para a cópia e o índice principal segue intacto | Verificação 6 |

In [1]:
# Os notebooks rodam a partir da raiz do projeto: é lá que ficam o .env, a pasta data/ e o pacote src/.
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Chat e embeddings vêm do Azure OpenAI configurado no .env (veja src/config.py).
from src.config import settings

## 0. Preparação (fornecido)

- `CachedQueryEmbeddings` (aula 3.4) evita pagar duas vezes o embedding da mesma pergunta nas avaliações de antes
  e depois;
- a cópia do índice vai para uma pasta temporária do sistema (`tempfile.mkdtemp`);
- `ask` roda uma pergunta no grafo corretivo e devolve resposta, fontes e consumo;
- guardamos o registro do índice principal (`index_info()`) para provar no fim que ele não mudou.

In [2]:
import shutil
import tempfile
from pathlib import Path

import pandas as pd
from langchain_core.embeddings import Embeddings

from src.evaluation import evaluate_retrieval, load_questions
from src.graph.corrective_rag import build_rag_graph
from src.models import get_embeddings
from src.rag.ingestion import estimate_tokens, index_incremental, load_folder, split_documents
from src.rag.retrieval import HybridRetriever
from src.rag.vectorstore import index_exists, index_info, load_index

pd.set_option("display.max_colwidth", 90)


class CachedQueryEmbeddings(Embeddings):
    """Reaproveita o vetor de consultas repetidas; os documentos passam direto para o modelo."""

    def __init__(self, base: Embeddings):
        self.base = base
        self.cache: dict[str, list[float]] = {}

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.base.embed_documents(texts)

    def embed_query(self, text: str) -> list[float]:
        if text not in self.cache:
            self.cache[text] = self.base.embed_query(text)
        return self.cache[text]


if not index_exists():
    from src.rag.ingestion import index_folder

    index_folder()

embeddings = CachedQueryEmbeddings(get_embeddings())
MAIN_INDEX_INFO = index_info()  # registro do índice principal, para conferir no fim
NEW_DOCUMENTS_DIR = Path("data/new_documents")
FIRMWARE_QUESTION = "O que muda na versão 1.3.0 do firmware do Fone Nova Buds?"

work_dir = Path(tempfile.mkdtemp(prefix="technova_final_project_"))
index_copy = work_dir / "faiss_index"
shutil.copytree(settings.index_dir, index_copy)


def make_retriever(folder: Path) -> HybridRetriever:
    """Retriever do produto sobre o índice de `folder` (com o cache de consultas)."""
    return HybridRetriever(index=load_index(folder, embeddings=embeddings))


def ask(graph, question: str) -> dict:
    out = graph.invoke({"messages": [("user", question)]})
    return {
        "answer": out["messages"][-1].content,
        "found": out["found"],
        "sources": [s["source"] for s in out["sources"]],
        "usage": out["usage"],
    }


print("Índice principal:", MAIN_INDEX_INFO)
print("Cópia de trabalho:", index_copy)

2026-09-27 22:19:06 | INFO     | src.models | Inicializando embeddings | provedor=azure | deployment=text-embedding-3-large


Índice principal: {'embeddings': 'azure · text-embedding-3-large', 'chunks': 321, 'dimensions': 3072, 'updated_at': '2026-09-27T20:39:15'}
Cópia de trabalho: /private/tmp/claude-501/-Users-mitoura-Documents-LangGraph/27ed4402-274d-4728-9753-90c38a175204/scratchpad/m5rag/tmp/technova_final_project_9zqgwvq9/faiss_index


Antes de indexar, veja o que vai entrar (sem chamar nenhuma API):

In [3]:
new_documents = load_folder(NEW_DOCUMENTS_DIR)
new_chunks = split_documents(new_documents)
NEW_SOURCES = sorted({d.metadata["source"] for d in new_documents})

print(f"{len(new_documents)} documentos, {len(new_chunks)} trechos, ~{estimate_tokens(new_chunks)} tokens de embeddings")
pd.DataFrame([
    {"source": s, "title": next(d.metadata["title"] for d in new_documents if d.metadata["source"] == s),
     "chunks": sum(c.metadata["source"] == s for c in new_chunks)}
    for s in NEW_SOURCES
])

2026-09-27 22:19:06 | INFO     | src.rag.ingestion | Documentos carregados | arquivos=3 | documentos=7


2026-09-27 22:19:06 | INFO     | src.rag.ingestion | Documentos divididos | documentos=7 | trechos=30


7 documentos, 30 trechos, ~5698 tokens de embeddings


,source,title,chunks
0,nova_buds_firmware_update.md,Comunicado de Atualização de Firmware do Fone Nova Buds (versão 1.3.0),11
1,nova_cinema_soundbar.html,"Soundbar Nova Cinema: instalação, conexões e solução de problemas | Central de Ajuda T...",5
2,sm-005_nova_fold.pdf,Manual do usuário: Smartphone Nova Fold,14


> ⚠️ Repare nos nomes das fontes: `nova_buds_firmware_update.md`, sem pasta. O `source` é o caminho **relativo à
> pasta indexada** (`data/new_documents`). Isso importa na hora de tornar a inclusão permanente (seção final).

## Atividade 1: a pergunta do firmware, antes e depois

### 1a. Antes

Crie o retriever sobre a **cópia** (ainda sem os documentos novos), monte o grafo corretivo com ele e faça a
pergunta `FIRMWARE_QUESTION`. O manual do Nova Buds só cita o firmware de fábrica (1.2.0): o esperado é o
assistente admitir que não sabe.

In [4]:
retriever_before = make_retriever(index_copy)
graph_before = build_rag_graph(retriever_before)
before = ask(graph_before, FIRMWARE_QUESTION)
print(before["answer"], "\nFontes:", before["sources"], "| found:", before["found"])

2026-09-27 22:19:06 | INFO     | src.graph.corrective_rag | Pergunta do turno | pergunta=O que muda na versão 1.3.0 do firmware do Fone Nova Buds?


2026-09-27 22:19:07 | INFO     | src.models | Inicializando LLM | provedor=azure | deployment=gpt-5.4-mini


2026-09-27 22:19:09 | INFO     | src.graph.corrective_rag | Avaliação | recuperados=6 | relevantes=6


Não encontrei essa informação nos documentos da TechNova. Se quiser, posso encaminhar sua dúvida para um atendente. 
Fontes: [] | found: False


In [5]:
# Verificação 1: antes da inclusão
assert isinstance(before, dict) and {"answer", "found", "sources"} <= set(before), "Use a função ask para montar `before`."
assert not set(before["sources"]) & set(NEW_SOURCES), "Os documentos novos não deveriam estar na cópia ainda!"
assert before["found"] is False, (
    "O assistente respondeu como se soubesse. Leia a resposta: se ela só cita o firmware 1.2.0 do manual, o grafo "
    "aprovou trechos do manual; rode de novo ou anote o caso na análise."
)
print("✅ Antes da inclusão, o assistente admite que não sabe sobre a versão 1.3.0.")

✅ Antes da inclusão, o assistente admite que não sabe sobre a versão 1.3.0.


### 1b. Incluir os documentos na cópia

Use `index_incremental(folder, target)` de [src/rag/ingestion.py](../src/rag/ingestion.py): ele carrega a pasta,
divide, adiciona **só os trechos cujo id ainda não está no índice** e salva no `target`. Passe a **cópia** como
`target`: sem ele, a função grava no índice principal.

In [6]:
report = index_incremental(NEW_DOCUMENTS_DIR, index_copy)
print(report)

2026-09-27 22:19:10 | INFO     | src.rag.ingestion | Documentos carregados | arquivos=3 | documentos=7


2026-09-27 22:19:10 | INFO     | src.rag.ingestion | Documentos divididos | documentos=7 | trechos=30


2026-09-27 22:19:11 | INFO     | src.rag.ingestion | Trechos adicionados | novos=30 | ignorados=0


2026-09-27 22:19:11 | INFO     | src.rag.vectorstore | Índice salvo | pasta=/private/tmp/claude-501/-Users-mitoura-Documents-LangGraph/27ed4402-274d-4728-9753-90c38a175204/scratchpad/m5rag/tmp/technova_final_project_9zqgwvq9/faiss_index | trechos=351


{'chunks_read': 30, 'added': 30, 'total_in_index': 351}


In [7]:
# Verificação 2: cópia atualizada, índice principal intacto
assert report["added"] == len(new_chunks), f"Esperava {len(new_chunks)} trechos novos; entraram {report['added']}."
assert index_info(index_copy)["chunks"] == MAIN_INDEX_INFO["chunks"] + len(new_chunks)
assert index_info() == MAIN_INDEX_INFO, "O índice principal mudou! Você esqueceu o target?"
print(f"✅ Cópia com {index_info(index_copy)['chunks']} trechos; índice principal intacto ({MAIN_INDEX_INFO['chunks']}).")

✅ Cópia com 351 trechos; índice principal intacto (321).


> 💡 **Dica:** rode a célula de inclusão de novo e veja `added: 0`. Os ids estáveis (hash de fonte + conteúdo)
> tornam a inclusão **idempotente**: repetir não duplica trechos.

### 1c. Depois

O `HybridRetriever` monta o BM25 na criação e o FAISS foi salvo em disco: crie um retriever **novo** a partir da
cópia, um grafo novo, e pergunte de novo.

In [8]:
retriever_after = make_retriever(index_copy)
graph_after = build_rag_graph(retriever_after)
after = ask(graph_after, FIRMWARE_QUESTION)
print(after["answer"], "\nFontes:", after["sources"], "| found:", after["found"])

2026-09-27 22:19:11 | INFO     | src.graph.corrective_rag | Pergunta do turno | pergunta=O que muda na versão 1.3.0 do firmware do Fone Nova Buds?


2026-09-27 22:19:12 | INFO     | src.graph.corrective_rag | Avaliação | recuperados=6 | relevantes=4


Na versão 1.3.0 do firmware do Fone Nova Buds, houve: correção das desconexões intermitentes no modo multiponto e em ambientes com interferência; troca entre dispositivos do multiponto mais rápida (redução média de 40% no tempo de troca); ajuste do indicador de bateria do estojo no app, que em alguns casos mostrava 10% a mais do que a carga real; novo aviso sonoro quando a bateria dos fones chega a 10%; e melhoria na estabilidade do microfone em chamadas com o ANC ligado. A autonomia informada no manual não muda: até 6 horas com ANC ligado, 8 horas desligado e até 24 horas no total com o estojo, com ANC ligado. [1][3] 
Fontes: ['nova_buds_firmware_update.md', 'nova_buds_firmware_update.md'] | found: True


In [9]:
# Verificação 3: depois da inclusão
assert after["found"] is True, "Depois da inclusão, o assistente deveria responder."
assert "nova_buds_firmware_update.md" in after["sources"], f"A resposta deveria citar o comunicado; citou {after['sources']}."
assert "1.3.0" in after["answer"] or "multiponto" in after["answer"].lower(), "A resposta não fala da versão nova."
print("✅ Depois da inclusão, a resposta cita o comunicado do firmware.")
print(f"Consumo: antes {before['usage'].get('total_tokens')} tokens | depois {after['usage'].get('total_tokens')} tokens")

✅ Depois da inclusão, a resposta cita o comunicado do firmware.
Consumo: antes 3583 tokens | depois 2570 tokens


## Atividade 2: perguntas novas de avaliação

Escreva 3 perguntas, **uma por documento novo**, no formato de `data/eval/questions.json`: `id`, `type` (`fact`,
`procedure` ou `multi_document`), `question` (como um cliente escreveria), `reference_answer` (a resposta certa,
tirada do documento) e `expected_sources` (use os nomes de `NEW_SOURCES`).

Depois meça a recuperação das 3 perguntas com a configuração do produto (`retriever.search` padrão) **antes**
(`retriever_before`) e **depois** (`retriever_after`).

<details><summary>Dica</summary>

Abra os arquivos em `data/new_documents/` e procure um dado pontual (versão, tamanho de tela, tipo de conexão) ou
um passo a passo. Evite copiar a frase do documento: o cliente usa outras palavras, e é isso que a avaliação precisa
testar. Para medir: `evaluate_retrieval(retriever_before.search, NEW_QUESTIONS)`.
</details>

In [10]:
NEW_QUESTIONS = [
    {
        "id": "n01",
        "type": "fact",
        "question": "Meu Nova Buds fica desconectando quando uso com o celular e o notebook ao mesmo tempo. Tem alguma atualização que resolve?",
        "reference_answer": "Sim. A versão 1.3.0 do firmware, disponível desde 22/09/2026 pelo app Nova Connect, corrige as desconexões intermitentes no modo multiponto.",
        "expected_sources": ["nova_buds_firmware_update.md"],
    },
    {
        "id": "n02",
        "type": "procedure",
        "question": "Como eu ligo a soundbar Nova Cinema na TV pra ter o áudio 3D e controlar o volume pelo controle da TV?",
        "reference_answer": "Pela conexão HDMI eARC: com os aparelhos desligados, ligue o cabo HDMI na porta eARC da soundbar e na porta HDMI eARC (ou ARC) da TV.",
        "expected_sources": ["nova_cinema_soundbar.html"],
    },
    {
        "id": "n03",
        "type": "fact",
        "question": "De quantas polegadas é a tela de dentro do Nova Fold?",
        "reference_answer": "A tela interna do Smartphone Nova Fold é OLED de 7,6 polegadas, a 120 Hz.",
        "expected_sources": ["sm-005_nova_fold.pdf"],
    },
]

new_table_before, new_before = evaluate_retrieval(retriever_before.search, NEW_QUESTIONS)
new_table_after, new_after = evaluate_retrieval(retriever_after.search, NEW_QUESTIONS)
print("Antes :", new_before)
print("Depois:", new_after)
new_table_after[["id", "hit", "rr", "retrieved"]]

Antes : {'questions': 3, 'hit_rate': 0.0, 'mrr': 0.0, 'coverage': 0.0}
Depois: {'questions': 3, 'hit_rate': 1.0, 'mrr': 1.0, 'coverage': 1.0}


,id,hit,rr,retrieved
0,n01,1,1.0,"[nova_buds_firmware_update.md, nova_buds_firmware_update.md, nova_buds_firmware_update..."
1,n02,1,1.0,"[nova_cinema_soundbar.html, nova_cinema_soundbar.html, nova_cinema_soundbar.html, nova..."
2,n03,1,1.0,"[sm-005_nova_fold.pdf, sm-005_nova_fold.pdf, sm-005_nova_fold.pdf, sm-005_nova_fold.pdf]"


In [11]:
# Verificação 4: perguntas novas
REQUIRED_KEYS = {"id", "type", "question", "reference_answer", "expected_sources"}
assert len(NEW_QUESTIONS) == 3, "Escreva exatamente 3 perguntas."
for item in NEW_QUESTIONS:
    assert REQUIRED_KEYS <= set(item), f"Faltam chaves em {item.get('id')}: {REQUIRED_KEYS - set(item)}"
    assert item["type"] in {"fact", "procedure", "multi_document"}, f"{item['id']}: tipo inválido ({item['type']})."
    assert isinstance(item["question"], str) and len(item["question"]) > 15, f"{item['id']}: escreva a pergunta."
    assert set(item["expected_sources"]) & set(NEW_SOURCES), f"{item['id']}: a fonte esperada deve ser um documento novo {NEW_SOURCES}."
covered = {s for item in NEW_QUESTIONS for s in item["expected_sources"]}
assert covered >= set(NEW_SOURCES), f"Faça uma pergunta por documento novo; faltou {set(NEW_SOURCES) - covered}."
assert new_before["hit_rate"] == 0, "Antes da inclusão, nenhuma pergunta nova deveria achar a fonte."
assert new_after["hit_rate"] >= 2 / 3, (
    f"Depois da inclusão, esperava hit rate >= 0,67; veio {new_after['hit_rate']}. Veja a coluna retrieved: "
    "a pergunta usa palavras que aparecem no documento?"
)
print(f"✅ Perguntas novas: hit rate {new_before['hit_rate']} -> {new_after['hit_rate']} | MRR {new_before['mrr']} -> {new_after['mrr']}")

✅ Perguntas novas: hit rate 0.0 -> 1.0 | MRR 0.0 -> 1.0


## Atividade 3: regressão nas perguntas antigas

Documentos novos disputam as mesmas vagas de `k` com os antigos. E há um efeito menos óbvio: o BM25 recalcula as
estatísticas das palavras (o IDF) com o corpus novo, então **mesmo perguntas sem relação** com os documentos novos
podem mudar de ordem. Rode a avaliação completa (`evaluate_retrieval` sem `questions`, as 26 antigas) antes e
depois e compare **pergunta a pergunta**.

In [12]:
old_table_before, old_before = evaluate_retrieval(retriever_before.search)
old_table_after, old_after = evaluate_retrieval(retriever_after.search)

regression = old_table_before[["id", "type", "hit", "rr", "coverage"]].merge(
    old_table_after[["id", "hit", "rr", "coverage", "retrieved"]], on="id", suffixes=("_before", "_after")
)
changed = regression[(regression["rr_before"] != regression["rr_after"]) | (regression["coverage_before"] != regression["coverage_after"])]
print("Antes :", old_before)
print("Depois:", old_after)
changed

Antes : {'questions': 26, 'hit_rate': 0.923, 'mrr': 0.683, 'coverage': 0.904}
Depois: {'questions': 26, 'hit_rate': 0.923, 'mrr': 0.657, 'coverage': 0.885}


,id,type,hit_before,rr_before,coverage_before,hit_after,rr_after,coverage_after,retrieved
21,q22,multi_document,1,1.0,1.0,1,0.333333,0.5,"[manuals/fn-002_nova_studio.pdf, manuals/fn-002_nova_studio.pdf, policies/shipping.md,..."


In [13]:
# Verificação 5: nada piorou (com tolerância explícita)
MRR_TOLERANCE = 0.05
lost_hit = regression[(regression["hit_before"] == 1) & (regression["hit_after"] == 0)]
assert old_before["questions"] == old_after["questions"] == 26, "Avalie as 26 perguntas antigas nos dois lados."
assert lost_hit.empty, f"Perguntas antigas que perderam o acerto: {lost_hit['id'].tolist()}. Investigue antes de publicar."
assert old_after["hit_rate"] >= old_before["hit_rate"], "O hit rate das perguntas antigas caiu."
assert old_before["mrr"] - old_after["mrr"] <= MRR_TOLERANCE, f"O MRR caiu mais que {MRR_TOLERANCE}."
print(f"✅ Regressão: hit rate {old_before['hit_rate']} -> {old_after['hit_rate']} | MRR {old_before['mrr']} -> {old_after['mrr']} "
      f"| {len(changed)} pergunta(s) mudaram de posição, nenhuma perdeu o acerto.")

✅ Regressão: hit rate 0.923 -> 0.923 | MRR 0.683 -> 0.657 | 1 pergunta(s) mudaram de posição, nenhuma perdeu o acerto.


**Analise** as perguntas em `changed`: nenhuma perdeu o acerto, mas alguma fonte esperada pode ter descido na
lista. Pergunte: o documento novo que entrou no topo **também responde** a pergunta (como a FAQ na aula 3.4) ou é
ruído? Uma queda pequena de MRR com hit rate intacto costuma ser aceitável; uma pergunta que perde o acerto não é.
Por isso a tolerância está escrita no código: a decisão de publicar fica registrada, não na memória de alguém.

## Atividade 4: produção

### 4a. Rodar o app com a cópia

O `app.py` (Streamlit) usa o grafo corretivo com checkpointer SQLite e lê o índice de `settings.index_dir`, que vem
da variável de ambiente `INDEX_DIR`. Dá para testar o app com a cópia **sem tocar no índice principal**:

```bash
INDEX_DIR=<pasta da cópia> streamlit run app.py
```

Monte o comando com o caminho da cópia em `app_command` e confira mais uma vez que o índice principal está intacto.

In [14]:
app_command = f"INDEX_DIR={index_copy} streamlit run app.py"
print("Para testar o app com a cópia, na raiz do projeto:\n ", app_command)
print("Para rodar com o índice principal:\n  streamlit run app.py")

Para testar o app com a cópia, na raiz do projeto:
  INDEX_DIR=/private/tmp/claude-501/-Users-mitoura-Documents-LangGraph/27ed4402-274d-4728-9753-90c38a175204/scratchpad/m5rag/tmp/technova_final_project_9zqgwvq9/faiss_index streamlit run app.py
Para rodar com o índice principal:
  streamlit run app.py


In [15]:
# Verificação 6: comando do app e índice principal
assert isinstance(app_command, str) and "streamlit run app.py" in app_command, "O comando deve rodar streamlit run app.py."
assert f"INDEX_DIR={index_copy}" in app_command, "Aponte INDEX_DIR para a cópia (index_copy)."
assert index_info() == MAIN_INDEX_INFO, "O índice principal mudou durante o projeto!"
print("✅ App pronto para testar com a cópia; índice principal intacto:", MAIN_INDEX_INFO["updated_at"])

✅ App pronto para testar com a cópia; índice principal intacto: 2026-09-27T20:39:15


No app, faça a pergunta do firmware e confira as fontes no painel da resposta, e o consumo de tokens em "Como a
resposta foi montada". Teste também as perguntas do README (troca, garantia, a armadilha do cupom NOVA50 e a placa
de vídeo que a base não tem).

> ⚠️ O app grava as conversas em `data/checkpoints.db`. Para recomeçar, apague esse arquivo.

### 4b. Tornar a inclusão permanente

Com a evidência em mãos (atividades 1 a 3), a inclusão no produto são dois passos, na raiz do projeto:

```bash
# 1. adiciona os documentos novos ao índice principal, sem recriar o resto
python -m src.rag.ingestion data/new_documents --incremental

# 2. move os arquivos para a base, para que a próxima reindexação completa os inclua
mv data/new_documents/nova_buds_firmware_update.md data/documents/
mv data/new_documents/nova_cinema_soundbar.html data/documents/help_center/
mv data/new_documents/sm-005_nova_fold.pdf data/documents/manuals/
```

E acrescente as 3 perguntas novas a `data/eval/questions.json`, para que toda avaliação futura as inclua.

> ⚠️ **Armadilha comum:** o `source` é relativo à pasta indexada. No passo 1 os trechos entram como
> `sm-005_nova_fold.pdf`; depois de mover, uma reindexação completa (`python -m src.rag.ingestion`) gera
> `manuals/sm-005_nova_fold.pdf`, com ids novos. Não rode `--incremental` sobre `data/documents` com os trechos
> antigos ainda no índice: os mesmos documentos entrariam de novo, com outro nome. Duas saídas limpas: reindexar
> tudo depois de mover (custo baixo, veja a aula 3.5) ou **mover primeiro** e rodar
> `python -m src.rag.ingestion data/documents --incremental`, que só adiciona o que é novo e já usa os nomes
> definitivos. Em qualquer caso, as `expected_sources` das perguntas novas precisam usar o nome final.

## Desafio opcional

1. Escreva uma pergunta `multi_document` que misture um documento novo e um antigo, por exemplo: "Atualizei o Nova
   Buds para a 1.3.0 e ele continua desconectando. Ainda estou na garantia?" (fontes: o comunicado e
   `policies/warranty.md`). Meça antes e depois e rode no `graph_after`.
2. Use `remove_source(index, source)` em uma cópia carregada em memória para tirar o manual do Nova Fold e veja a
   pergunta `n03` voltar a falhar. É assim que se despublica um documento.
3. Julgue a resposta do firmware com `judge_answer` (aula 3.5), usando a sua `reference_answer`.

## Limpeza

Quando terminar de testar o app com a cópia, apague a pasta temporária:

In [16]:
shutil.rmtree(work_dir, ignore_errors=True)
print("Pasta temporária removida:", not work_dir.exists(), "| índice principal intacto:", index_info() == MAIN_INDEX_INFO)

Pasta temporária removida: True | índice principal intacto: True


## Resumo

- Incluir documentos é uma **mudança de produto**: faça em uma cópia do índice e publique só com evidência.
- `index_incremental(folder, target)` adiciona só trechos novos (ids estáveis), sem recriar o índice; sem `target`,
  grava no índice principal.
- Antes x depois na pergunta de negócio (firmware 1.3.0): de "não encontrei" para uma resposta que cita o comunicado.
- Perguntas novas de avaliação garantem que a busca **encontra** os documentos novos; a regressão nas 26 antigas
  garante que **nada piorou**, com a tolerância escrita no código.
- Adicionar documentos muda até perguntas sem relação (competição por `k` e estatísticas do BM25): compare
  pergunta a pergunta, não só a média.
- O app roda com `streamlit run app.py`; `INDEX_DIR` aponta para outro índice sem mexer no principal. Para tornar
  permanente: ingestão incremental, mover os arquivos para `data/documents` (atenção ao `source`) e atualizar o
  conjunto de avaliação.

## O que vem depois

Você terminou o curso com um assistente de RAG completo: ingestão, busca híbrida com filtros, respostas com
citações, grafo corretivo, agente, avaliação e cuidados de produção. Alguns caminhos para continuar (só para
conhecer):

| Tema | Ideia |
|---|---|
| **RAG multimodal** | indexar imagens, tabelas e diagramas dos manuais (legendas geradas por modelos de visão ou embeddings multimodais), não só o texto extraído |
| **Bases vetoriais gerenciadas** | trocar o FAISS local por um serviço (pgvector, Azure AI Search, Pinecone, Qdrant...) com filtros no servidor, escala e atualização concorrente; no projeto, a troca fica isolada em `src/rag/vectorstore.py` |
| **Avaliação contínua** | rodar a avaliação a cada mudança (CI), coletar perguntas reais e feedback dos usuários, e acompanhar as métricas ao longo do tempo |
| **Fine-tuning de embeddings** | ajustar um modelo de embeddings com pares pergunta x trecho do seu domínio quando os modelos genéricos não entendem o vocabulário (códigos, jargões internos) |